# 2SFCA: Maps and Diagnostics

Visualizes `tess_all_access.csv`. Run `2sfca_compute.ipynb` first.
Maps render in EPSG:3857; scalebars use dx = 1 m (projected units).
Figures save to `notebooks/figures/2sfca/`.

Tier columns are not in the CSV (they are compute-notebook diagnostics), so tier breakdowns in `summarize_highlight` are skipped automatically.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import geopandas as gpd
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from src.crs import CRS_WGS84
from src.io import ensure_dir
from src.paths import FIGURES, PHARMACIES_MASTER, SAL_W_WARD_DEDUP, TESS_ALL_ACCESS
from src.plotting import FONT_BODY, FONT_TITLE, setup_fonts

setup_fonts()
FIG_DIR = ensure_dir(FIGURES / "2sfca")

In [ ]:
allaccess = pd.read_csv(TESS_ALL_ACCESS, low_memory=False)
allaccess["EA_CODE"] = pd.to_numeric(allaccess["EA_CODE"], errors="coerce").astype("Int64")

sal_polygons = gpd.read_file(SAL_W_WARD_DEDUP)
sal_polygons["EA_CODE"] = pd.to_numeric(sal_polygons["EA_CODE"], errors="coerce").astype("Int64")

pharmacy_raw = pd.read_csv(PHARMACIES_MASTER)
pharmacy_gdf = gpd.GeoDataFrame(
    pharmacy_raw,
    geometry=gpd.points_from_xy(pharmacy_raw["LNG"], pharmacy_raw["LAT"]),
    crs=CRS_WGS84,
)

# Web Mercator for display.
sal_polygons_3857 = sal_polygons.to_crs(epsg=3857)
sal_polygons_dissolved = sal_polygons_3857.dissolve(by="EA_CODE").reset_index()

allaccess = allaccess.drop(columns=["geometry"], errors="ignore").merge(
    sal_polygons_dissolved[["EA_CODE", "geometry"]],
    on="EA_CODE",
    how="left",
)
allaccess = gpd.GeoDataFrame(allaccess, geometry="geometry", crs=sal_polygons_3857.crs)
pharm_with_sal = pharmacy_gdf.to_crs(allaccess.crs)

print(f"MAP DATA READY: {len(allaccess)} SALs")

In [ ]:
# Province overview: SAL boundaries with pharmacy locations.
kzn = allaccess[allaccess["PR_NAME"] == "KwaZulu-Natal"]
gau = allaccess[allaccess["PR_NAME"] == "Gauteng"]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_alpha(0.0)

for ax, prov_gdf, name in [(axes[0], kzn, "KwaZulu-Natal"), (axes[1], gau, "Gauteng")]:
    prov_gdf.plot(ax=ax, color="#dac291", edgecolor="#f9eece", linewidth=0.3)
    pharm_with_sal.cx[prov_gdf.total_bounds[0]:prov_gdf.total_bounds[2],
                      prov_gdf.total_bounds[1]:prov_gdf.total_bounds[3]].plot(
        ax=ax, color="#2e3e6c", markersize=3, alpha=1)
    ax.set_title(f"{name}: SALs and Pharmacies", fontfamily=FONT_TITLE, fontweight="bold")
    ax.set_facecolor("none")
    ax.axis("off")

plt.tight_layout()
fig.savefig(FIG_DIR / "2sfca_province_preview.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
# Score distribution histograms.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
formatter = FuncFormatter(lambda x, p: f"{int(x):,}")

panels = [
    (axes[0, 0], kzn, "Ai_walk", "#2e3e6c", "KZN Walk Accessibility ($A_{i}$)"),
    (axes[0, 1], kzn, "Ai_drive", "#ffcb05", "KZN Drive Accessibility ($A_{i}$)"),
    (axes[1, 0], gau, "Ai_walk", "#2e3e6c", "Gauteng Walk Accessibility ($A_{i}$)"),
    (axes[1, 1], gau, "Ai_drive", "#ffcb05", "Gauteng Drive Accessibility ($A_{i}$)"),
]
for ax, df, col, color, title in panels:
    df[col].hist(ax=ax, bins=50, color=color, edgecolor="white")
    ax.yaxis.set_major_formatter(formatter)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Accessibility Index ($A_{i}$)", fontweight="bold")

plt.suptitle("Accessibility Score Distributions by Province and Mode", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "A_i_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Color ramp shared across accessibility heatmaps.
access_cmap = LinearSegmentedColormap.from_list(
    "access",
    ["#2e3e6c", "#007264", "#ffcb05"]
)

# Single source of truth for city marker coordinates (EPSG:3857).
# Used by every highlight map below, the helper drops any label
# whose coordinates fall outside the rendered viewport.
CITIES = {
    "Johannesburg": (3118870, -3005460),
    "Pretoria": (3142400, -2952500),
    "Merafong City":(3049000, -3024000),
    "Durban": (3453460, -3488260),
    "Umhlanga": (3461500, -3452000),
    "Empangeni": (3562000, -3336000),
    "Msinga": (3391030, -3337800),
    "Ndwedwe": (3446500, -3421000),
    "Vulamehlo": (3409300, -3516000),
    "KwaMashu": (3449000, -3465000),
}

# Fixed city extents in EPSG:3857. Defined once and reused by the
# single-city maps and the dual side-by-side panel so all three views
# of Johannesburg (or Durban) are framed identically.
CITY_EXTENTS = {
    "Johannesburg": (3050000, 3170000, -3050000, -2950000),
    "Durban": (3420000, 3490000, -3510000, -3445000),
}


def compute_extent(subset_gdf, pad_pct = 0.20, min_span_m = 8000):
    """Return a padded bounding box for a subset GeoDataFrame.

    Pads symmetrically by pad_pct of the larger dimension so very
    elongated subsets do not produce sliver maps. Enforces a minimum
    span to keep tightly-clustered subsets from rendering at absurd
    zoom levels.
    """
    minx, miny, maxx, maxy = subset_gdf.total_bounds
    width = maxx - minx
    height = maxy - miny
    pad = pad_pct * max(width, height)
    pad = max(pad, min_span_m / 2)
    return (minx - pad, maxx + pad, miny - pad, maxy + pad)


def figsize_from_extent(xmin, xmax, ymin, ymax, base_in = 8):
    """Figure dimensions matching the extent's aspect ratio.

    Keeps the longer side at base_in so maps never exceed that
    size, while the shorter side scales down proportionally.
    """
    w = xmax - xmin
    h = ymax - ymin
    if w >= h:
        return (base_in, base_in * h / w)
    return (base_in * w / h, base_in)


def visible_cities(extent, names = None):
    """Yield (x, y, name) for cities falling inside the viewport.

    Drops any city whose coordinates are outside the visible
    extent so off-screen labels never widen the saved figure
    when bbox_inches='tight' is used.
    """
    xmin, xmax, ymin, ymax = extent
    candidates = names if names is not None else list(CITIES.keys())
    for name in candidates:
        x, y = CITIES[name]
        if xmin <= x <= xmax and ymin <= y <= ymax:
            yield x, y, name


def plot_highlight_map(
    province_gdf,
    highlight_codes,
    pharm_gdf,
    title,
    out_basename,
    city_names = None,
    show_pharmacies = True,
    pad_pct = 0.20,
):
    """Render a province map zoomed to a set of highlighted SALs.

    Steps:
      1. Compute extent from the highlighted subset only.
      2. Match figure aspect to that extent.
      3. Clip province polygons and pharmacy points to the viewport
         before plotting (avoids drawing tens of thousands of unseen
         polygons and prevents off-screen elements from inflating the
         saved figure).
      4. Draw only the city labels that fall inside the viewport.
      5. Save both opaque and transparent PNGs at 300 dpi.
    """
    subset = province_gdf[province_gdf["EA_CODE"].isin(highlight_codes)]
    if len(subset) == 0:
        print(f"WARNING: NO MATCHING SALS FOR {title}")
        return

    extent = compute_extent(subset, pad_pct = pad_pct)
    xmin, xmax, ymin, ymax = extent
    figsize = figsize_from_extent(xmin, xmax, ymin, ymax, base_in = 8)

    # Clip province and pharmacy layers to the viewport.
    prov_clip = province_gdf.cx[xmin:xmax, ymin:ymax]
    pharm_clip = pharm_gdf.cx[xmin:xmax, ymin:ymax] if show_pharmacies else None

    fig, ax = plt.subplots(figsize = figsize)
    ax.set_facecolor("#fcffeb")

    # Draw layers in stacking order.
    prov_clip.plot(ax = ax, color = "#dac291", edgecolor = "#f9eece", linewidth = 0.5)
    subset.plot(ax = ax, color = "#d14b18", edgecolor = "#f9eece", linewidth = 0.5)
    if pharm_clip is not None and len(pharm_clip) > 0:
        pharm_clip.plot(ax = ax, color = "#2e3e6c", markersize = 4, alpha = 1.0, zorder = 5)

    # Draw city markers and labels for cities inside the viewport.
    for x, y, name in visible_cities(extent, names = city_names):
        ax.scatter(x, y, color = "black", s = 25, zorder = 20)
        ax.annotate(
            name,
            xy = (x, y),
            xytext = (6, 6),
            textcoords = "offset points",
            fontsize = 10,
            fontweight = "bold",
            fontname = FONT_BODY,
            zorder = 21,
        )

    # Lock aspect, lock viewport, scalebar, title.
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.add_artist(ScaleBar(dx = 1, units = "m", location = "lower right"))
    ax.set_title(title, fontsize = 13, fontweight = "bold", fontname = FONT_TITLE)
    ax.axis("off")

    fig.savefig(FIG_DIR / f"{out_basename}.png", dpi = 300, bbox_inches = "tight")
    fig.savefig(FIG_DIR / f"{out_basename}_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
    plt.show()

In [ ]:
# Frame on the shared Johannesburg extent so this view, the Durban view,
# and the dual side-by-side view all use the same physical map scale.
xmin, xmax, ymin, ymax = CITY_EXTENTS["Johannesburg"]
figsize = figsize_from_extent(xmin, xmax, ymin, ymax, base_in = 8)

# Clip data to the viewport before plotting.
subset_jhb = allaccess.cx[xmin:xmax, ymin:ymax]
pharm_jhb = pharm_with_sal.cx[xmin:xmax, ymin:ymax]

# Color stretched to the 2nd-98th percentile of in-view values.
vmin = subset_jhb["walk_log"].quantile(0.02)
vmax = subset_jhb["walk_log"].quantile(0.98)
norm = mpl.colors.Normalize(vmin = vmin, vmax = vmax)

fig, ax = plt.subplots(figsize = figsize)
ax.set_facecolor("#fcffeb")

subset_jhb.plot(column = "walk_log", cmap = access_cmap, norm = norm, ax = ax, edgecolor = "none")
pharm_jhb.plot(ax = ax, color = "#fcffeb", markersize = 6, alpha = 1.0, zorder = 5)

# City markers for any city inside the viewport.
extent = (xmin, xmax, ymin, ymax)
for x, y, name in visible_cities(extent, names = ["Johannesburg"]):
    ax.scatter(x, y, color = "#d14b18", s = 25, zorder = 20)
    ax.annotate(
        name,
        xy = (x, y),
        xytext = (8, 6),
        textcoords = "offset points",
        fontsize = 12,
        fontweight = "bold",
        fontname = FONT_BODY,
        zorder = 21,
        color = "#d14b18"
    )

# Lock aspect to data CRS units.
ax.set_aspect("equal")
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.add_artist(ScaleBar(dx = 1, units = "m", location = "upper left"))
ax.set_title("Walking Accessibility (Johannesburg)", fontweight = "bold", fontname = FONT_TITLE)
ax.axis("off")

# Colorbar.
sm = mpl.cm.ScalarMappable(cmap = access_cmap, norm = norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax = ax, fraction = 0.035, pad = 0.02)
cbar.set_label("Walking Accessibility (log)", fontweight = "bold", fontname = FONT_BODY)

plt.tight_layout()
fig.savefig(FIG_DIR / "A_i_johannesburg.png", dpi = 300, bbox_inches = "tight")
fig.savefig(FIG_DIR / "A_i_johannesburg_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
plt.show()

In [ ]:
# Frame on the shared Durban extent so single-city and dual views match.
xmin, xmax, ymin, ymax = CITY_EXTENTS["Durban"]
figsize = figsize_from_extent(xmin, xmax, ymin, ymax, base_in = 8)

# Clip to viewport.
subset_durban = allaccess.cx[xmin:xmax, ymin:ymax]
pharm_durban = pharm_with_sal.cx[xmin:xmax, ymin:ymax]

# Color stretch on in-view percentiles.
vmin = subset_durban["walk_log"].quantile(0.02)
vmax = subset_durban["walk_log"].quantile(0.98)
norm = mpl.colors.Normalize(vmin = vmin, vmax = vmax)

fig, ax = plt.subplots(figsize = figsize)
ax.set_facecolor("#fcffeb")

subset_durban.plot(column = "walk_log", cmap = access_cmap, norm = norm, ax = ax, edgecolor = "none")
pharm_durban.plot(ax = ax, color = "#fcffeb", markersize = 6, alpha = 1.0, zorder = 5)

# City markers for any city in view.
extent = (xmin, xmax, ymin, ymax)
for x, y, name in visible_cities(extent, names = ["Durban", "Umhlanga", "KwaMashu"]):
    ax.scatter(x, y, color = "#d14b18", s = 25, zorder = 20)
    ax.annotate(
        name,
        xy = (x, y),
        xytext = (8, 6),
        textcoords = "offset points",
        fontsize = 12,
        fontweight = "bold",
        fontname = FONT_BODY,
        zorder = 21,
        color = "#d14b18"
    )

ax.set_aspect("equal")
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.add_artist(ScaleBar(dx = 1, units = "m", location = "lower right"))
ax.set_title("Walking Accessibility (Durban)", fontweight = "bold", fontname = FONT_TITLE)
ax.axis("off")

sm = mpl.cm.ScalarMappable(cmap = access_cmap, norm = norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax = ax, fraction = 0.035, pad = 0.02)
cbar.set_label("Walking Accessibility (log)", fontweight = "bold", fontname = FONT_BODY)

plt.tight_layout()
fig.savefig(FIG_DIR / "A_i_durban.png", dpi = 300, bbox_inches = "tight")
fig.savefig(FIG_DIR / "A_i_durban_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
plt.show()

In [ ]:
# Side-by-side comparison at a single shared physical scale.
# width_ratios are set to the actual extent widths (in meters) so each
# panel covers the same number of kilometers per inch of figure space.
# Without this, matplotlib gives both panels equal axes width and the
# scalebars end up reading 25 km vs 15 km for the same screen distance.

extents = {name: CITY_EXTENTS[name] for name in ["Johannesburg", "Durban"]}
widths_m = [(xmax - xmin) for (xmin, xmax, _, _) in extents.values()]
heights_m = [(ymax - ymin) for (_, _, ymin, ymax) in extents.values()]

# Pick a target physical scale (meters per figure inch).
m_per_inch = 18000
panel_widths_in = [w / m_per_inch for w in widths_m]
panel_heights_in = [h / m_per_inch for h in heights_m]

# Figure: sum of panel widths plus padding for colorbar; height fits the tallest panel.
fig_w = sum(panel_widths_in) + 2.0
fig_h = max(panel_heights_in) + 1.0

fig, axes = plt.subplots(
    1, 2,
    figsize = (fig_w, fig_h),
    gridspec_kw = {"width_ratios": panel_widths_in}
)

# Shared color stretch across both city extents for fair comparison.
all_vals = pd.concat([
    allaccess.cx[xmin:xmax, ymin:ymax]["walk_log"]
    for (xmin, xmax, ymin, ymax) in extents.values()
])
vmin = all_vals.quantile(0.02)
vmax = all_vals.quantile(0.98)
norm = mpl.colors.Normalize(vmin = vmin, vmax = vmax)

for ax, (name, (xmin, xmax, ymin, ymax)) in zip(axes, extents.items()):
    subset = allaccess.cx[xmin:xmax, ymin:ymax]
    subset.plot(column = "walk_log", cmap = access_cmap, norm = norm, ax = ax, edgecolor = "none")

    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_facecolor("#fcffeb")
    ax.add_artist(ScaleBar(dx = 1, units = "m", location = "lower right"))
    ax.set_title(name, fontweight = "bold", fontname = FONT_TITLE)
    ax.axis("off")

# Single shared colorbar.
sm = mpl.cm.ScalarMappable(norm = norm, cmap = access_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax = axes, fraction = 0.025, pad = 0.02)
cbar.set_label("Walking Accessibility (log)", fontweight = "bold", fontname = FONT_BODY)

plt.suptitle("Walking Accessibility: Johannesburg vs. Durban", size = 14, fontweight = "bold", fontname = FONT_TITLE)

fig.savefig(FIG_DIR / "A_i_dual.png", dpi = 300, bbox_inches = "tight")
fig.savefig(FIG_DIR / "A_i_dual_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
plt.show()

In [ ]:
def summarize_highlight(gdf, highlight_codes, score_col, label):
    """Print descriptive stats for a set of highlighted SALs.

    Parameters
    ----------
    gdf : GeoDataFrame
        Province-level slice of allaccess (already filtered to one province).
    highlight_codes : list[int]
        EA_CODE values for the highlighted SALs.
    score_col : str
        Accessibility score column, either "Ai_walk" or "Ai_drive".
    label : str
        Human-readable label for the print header.
    """
    tier_col = score_col.replace("Ai_", "") + "_tier"
    subset = gdf[gdf["EA_CODE"].isin(highlight_codes)].copy()

    if len(subset) == 0:
        print(f"WARNING: no matching SALs for {label}")
        return

    # PER-SAL DETAIL TABLE.
    detail_cols = ["EA_CODE"]

    # Add place name if available.
    if "SP_NAME" in subset.columns:
        detail_cols.append("SP_NAME")

    detail_cols.append(score_col)

    if tier_col in subset.columns:
        detail_cols.append(tier_col)

    detail_cols.extend(["sal2023_est", "area_km2"])

    if "EA_TYPE" in subset.columns:
        detail_cols.append("EA_TYPE")
    if "econ_status" in subset.columns:
        detail_cols.append("econ_status")

    # Filter to columns that actually exist.
    detail_cols = [c for c in detail_cols if c in subset.columns]
    detail = subset[detail_cols].sort_values(score_col, ascending = False)

    print(f"{label}")
    print(f"SALs matched: {len(subset)}")
    print()
    print(detail.to_string(index = False))
    print()

    # AGGREGATE SUMMARY.
    total_pop = subset["sal2023_est"].sum()
    mean_score = subset[score_col].mean()
    median_score = subset[score_col].median()
    total_area = subset["area_km2"].sum()
    mean_density = total_pop / total_area if total_area > 0 else 0

    print(f"AGGREGATE")
    print(f"Total est. population (2023): {total_pop:,.0f}")
    print(f"Total area: {total_area:,.2f} km2")
    print(f"Mean pop density: {mean_density:,.0f} per km2")
    print(f"Mean {score_col}: {mean_score:.6f}")
    print(f"Median {score_col}: {median_score:.6f}")
    print()

    # SETTLEMENT TYPE BREAKDOWN.
    if "EA_GTYPE" in subset.columns:
        print("SETTLEMENT TYPE (EA_GTYPE)")
        print(subset["EA_GTYPE"].value_counts().to_string())
        print()

    # TIER BREAKDOWN.
    if tier_col in subset.columns:
        print(f"TIER DISTRIBUTION ({tier_col})")
        print(subset[tier_col].value_counts().to_string())
        print()

In [ ]:
queries = [
    ("KwaZulu-Natal", "Ai_walk", "KZN Bottom 10 Walk", False),
    ("KwaZulu-Natal", "Ai_walk", "KZN Top 10 Walk", True),
    ("KwaZulu-Natal", "Ai_drive", "KZN Bottom 10 Drive", False),
    ("KwaZulu-Natal", "Ai_drive", "KZN Top 10 Drive", True),
    ("Gauteng", "Ai_drive", "Gauteng Bottom 10 Drive", False),
    ("Gauteng", "Ai_drive", "Gauteng Top 10 Drive", True),
    ("Gauteng", "Ai_walk", "Gauteng Bottom 10 Walk", False),
    ("Gauteng", "Ai_walk", "Gauteng Top 10 Walk", True),
]

for prov_name, score_col, label, top in queries:
    cols = ["EA_CODE", score_col]
    if "SAL_NAME" in allaccess.columns:
        cols.append("SAL_NAME")
    df = allaccess[allaccess["PR_NAME"] == prov_name][cols].dropna(subset=[score_col])
    #df = df.drop_duplicates(subset="EA_CODE")
    result = df.nlargest(10, score_col) if top else df.nsmallest(10, score_col)
    codes = result["EA_CODE"].astype(int).tolist()

    print(f"{label}\n")
    print(result.to_string(index=False))
    print(f"\nhighlight_codes = {codes}\n\n")

In [ ]:
# KZN BOTTOM 10 WALK.
gdf_kzn = allaccess[allaccess["PR_NAME"] == "KwaZulu-Natal"]

plot_highlight_map(
    province_gdf = gdf_kzn,
    highlight_codes = [50310271, 50310262, 50310266, 50310265, 50310269,
                       50310270, 50310263, 50310264, 50310180, 50310249],
    pharm_gdf = pharm_with_sal,
    title = "KZN: Bottom 10 Walk $A_{i}$",
    out_basename = "bottom_10_KZN_walk",
    city_names = ["Durban", "Vulamehlo", "KwaMashu", "Umhlanga"],
    show_pharmacies = True
)

In [ ]:
# SUMMARY: KZN BOTTOM 10 WALK.
summarize_highlight(
    gdf = gdf_kzn,
    highlight_codes = [50310271, 50310262, 50310266, 50310265, 50310269,
                       50310270, 50310263, 50310264, 50310180, 50310249],
    score_col = "Ai_walk",
    label = "KZN: BOTTOM 10 WALK A_i"
)

In [ ]:
# KZN TOP 2 WALK.
# Selected because they are close enough together to view.
gdf_kzn = allaccess[allaccess["PR_NAME"] == "KwaZulu-Natal"]

plot_highlight_map(
    province_gdf = gdf_kzn,
    highlight_codes = [59913378, 59913388],
    pharm_gdf = pharm_with_sal,
    title = "KZN: Top 2 Walk $A_{i}$",
    out_basename = "top_2_KZN_walk",
    city_names = ["Durban", "Vulamehlo", "KwaMashu", "Umhlanga"],
    show_pharmacies = True,
)

In [ ]:
# SUMMARY: KZN TOP 2 WALK.
summarize_highlight(
    gdf = gdf_kzn,
    highlight_codes = [59913378, 59913388],
    score_col = "Ai_walk",
    label = "KZN: TOP 2 WALK A_i"
)

In [ ]:
# KZN BOTTOM 10 DRIVE.
# Same SALs as the walk view: every score in the drive bottom-10 is 0.0,
# and the tied SALs resolve to the same EA_CODEs as the walk version.
plot_highlight_map(
    province_gdf = gdf_kzn,
    highlight_codes = [50310271, 50310262, 50310266, 50310265, 50310269,
                       50310270, 50310263, 50310264, 50310180, 50310249],
    pharm_gdf = pharm_with_sal,
    title = "KZN: Bottom 10 Drive $A_{i}$",
    out_basename = "bottom_10_KZN_drive",
    city_names = ["Durban", "Vulamehlo", "KwaMashu", "Umhlanga"],
    show_pharmacies = False,
)

In [ ]:
# SUMMARY: KZN BOTTOM 10 DRIVE.
summarize_highlight(
    gdf = gdf_kzn,
    highlight_codes = [50310271, 50310262, 50310266, 50310265, 50310269,
                       50310270, 50310263, 50310264, 50310180, 50310249],
    score_col = "Ai_drive",
    label = "KZN: BOTTOM 10 DRIVE A_i"
)

In [ ]:
# KZN TOP 5 DRIVE.
plot_highlight_map(
    province_gdf = gdf_kzn,
    highlight_codes = [54210101, 54210224, 54210099, 54210230, 54210223],
    pharm_gdf = pharm_with_sal,
    title = "KZN: Top 5 Drive $A_{i}$",
    out_basename = "top_5_KZN_drive",
    city_names = ["Durban", "Vulamehlo", "KwaMashu", "Umhlanga"],
    show_pharmacies = False,
)

In [ ]:
# SUMMARY: KZN TOP 10 DRIVE.
summarize_highlight(
    gdf = gdf_kzn,
    highlight_codes = [54210101, 54210224, 54210099, 54210230, 54210223],
    score_col = "Ai_drive",
    label = "KZN: TOP 10 DRIVE A_i"
)

In [ ]:
# GAUTENG BOTTOM 10 DRIVE.
gdf_gau = allaccess[allaccess["PR_NAME"] == "Gauteng"]

plot_highlight_map(
    province_gdf = gdf_gau,
    highlight_codes = [76010966, 76010964, 76010981, 76010976, 76010975,
                       76010969, 76011312, 76010971, 76011134, 76010668],
    pharm_gdf = pharm_with_sal,
    title = "Gauteng: Bottom 10 Drive $A_{i}$",
    out_basename = "bottom_10_Gauteng_drive",
    city_names = ["Johannesburg", "Pretoria", "Merafong City"],
    show_pharmacies = True,
)

In [ ]:
# SUMMARY: GAUTENG BOTTOM 10 DRIVE.
summarize_highlight(
    gdf = gdf_gau,
    highlight_codes = [76010966, 76010964, 76010981, 76010976, 76010975,
                       76010969, 76011312, 76010971, 76011134, 76010668],
    score_col = "Ai_drive",
    label = "GAUTENG: BOTTOM 10 DRIVE A_i"
)

In [ ]:
# GAUTENG TOP 10 DRIVE.
gdf_gau = allaccess[allaccess["PR_NAME"] == "Gauteng"]

plot_highlight_map(
    province_gdf = gdf_gau,
    highlight_codes = [79814159, 79816032, 79814156, 79816031, 79814155,
                       79814153, 79814150, 79814154, 79816010, 79814152],
    pharm_gdf = pharm_with_sal,
    title = "Gauteng: Top 10 Drive $A_{i}$",
    out_basename = "top_10_Gauteng_drive",
    city_names = ["Johannesburg", "Pretoria", "Merafong City"],
    show_pharmacies = True,
)

In [ ]:
# SUMMARY: GAUTENG TOP 10 DRIVE.
summarize_highlight(
    gdf = gdf_gau,
    highlight_codes = [79814159, 79816032, 79814156, 79816031, 79814155,
                       79814153, 79814150, 79814154, 79816010, 79814152],
    score_col = "Ai_drive",
    label = "GAUTENG: TOP 10 DRIVE A_i"
)

In [ ]:
# GAUTENG BOTTOM 10 WALK.
plot_highlight_map(
    province_gdf = gdf_gau,
    highlight_codes = [76010955, 76011356, 76010940, 76010946, 76010952,
                       76011355, 76010947, 76011358, 76011357, 76010951],
    pharm_gdf = pharm_with_sal,
    title = "Gauteng: Bottom 10 Walk $A_{i}$",
    out_basename = "bottom_10_Gauteng_walk",
    city_names = ["Johannesburg", "Pretoria"],
    show_pharmacies = True,
)

In [ ]:
# SUMMARY: GAUTENG BOTTOM 10 WALK.
summarize_highlight(
    gdf = gdf_gau,
    highlight_codes = [76010955, 76011356, 76010940, 76010946, 76010952,
                       76011355, 76010947, 76011358, 76011357, 76010951],
    score_col = "Ai_walk",
    label = "GAUTENG: BOTTOM 10 WALK A_i"
)

In [ ]:
# GAUTENG TOP 10 WALK.
plot_highlight_map(
    province_gdf = gdf_gau,
    highlight_codes = [79710820, 79910649, 79910250, 79910810, 79815538,
                       79910793, 79911564, 76310210, 79815860, 79815868],
    pharm_gdf = pharm_with_sal,
    title = "Gauteng: Top 10 Walk $A_{i}$",
    out_basename = "top_10_Gauteng_walk",
    city_names = ["Johannesburg", "Pretoria"],
    show_pharmacies = True,
)

In [ ]:
# SUMMARY: GAUTENG TOP 10 WALK.
summarize_highlight(
    gdf = gdf_gau,
    highlight_codes = [79710820, 79910649, 79910250, 79910810, 79815538,
                       79910793, 79911564, 76310210, 79815860, 79815868],
    score_col = "Ai_walk",
    label = "GAUTENG: TOP 10 WALK A_i"
)

In [ ]:
# BIVARIATE WALKING vs DRIVING ACCESSIBILITY.
# Each SAL is binned into a 3x3 grid on terciles of log walk and log drive
# scores. Grey = low/low (most underserved). Insets zoom into Johannesburg
# and Durban using the shared CITY_EXTENTS so framing matches earlier maps.

# Data CRS is already EPSG:3857 from the data prep cell above.
# Tercile bins computed on the positive-score subset.
allaccess["walk_q"] = 0
mask = allaccess["walk_log"] > 0
allaccess.loc[mask, "walk_q"] = pd.qcut(allaccess.loc[mask, "walk_log"], 3, labels = [0, 1, 2])

allaccess["drive_q"] = 0
mask = allaccess["drive_log"] > 0
allaccess.loc[mask, "drive_q"] = pd.qcut(allaccess.loc[mask, "drive_log"], 3, labels = [0, 1, 2])

# 3x3 bivariate palette: grey corner low/low, blue corner high/high.
biv_colors = {
    (0, 0): "#E7E7E7",
    (0, 1): "#FFF2CC", (0, 2): "#FFC000",
    (1, 0): "#C6EFCE", (1, 1): "#92D18E", (1, 2): "#92D050",
    (2, 0): "#00784B", (2, 1): "#00A69B", (2, 2): "#005096",
}
allaccess["biv_color"] = allaccess.apply(
    lambda r: biv_colors[(int(r["walk_q"]), int(r["drive_q"]))],
    axis = 1
)
counts = pd.crosstab(allaccess["walk_q"], allaccess["drive_q"])

fig, ax = plt.subplots(figsize = (12, 11))
ax.set_facecolor("#fcffeb")
allaccess.plot(color = allaccess["biv_color"], ax = ax, linewidth = 0.05, edgecolor = "white")

# City labels visible at province scale.
for x, y, name in visible_cities(
    extent = ax.get_xlim() + ax.get_ylim(),
    names = ["Johannesburg", "Pretoria", "Durban"]
):
    ax.scatter(x, y, color = "black", s = 10, zorder = 20)
    ax.annotate(
        name,
        xy = (x, y),
        xytext = (8, 6),
        textcoords = "offset points",
        fontsize = 10,
        fontweight = "bold",
        fontname = FONT_BODY,
    )

ax.set_aspect("equal")
ax.set_title("Bivariate Accessibility: Walking vs. Driving", fontsize = 14, fontweight = "bold", fontname = FONT_TITLE)
ax.axis("off")
ax.add_artist(ScaleBar(dx = 1, units = "m", location = "lower left"))

# Bivariate legend grid in the lower-right.
legend_ax = fig.add_axes([0.78, 0.16, 0.16, 0.16])
legend_ax.set_facecolor("none")
legend_ax.axis("off")
for i in range(3):
    for j in range(3):
        rect = mpatches.Rectangle((j, i), 1, 1, facecolor = biv_colors[(i, j)], edgecolor = "white")
        legend_ax.add_patch(rect)
        value = counts.loc[i, j] if (i in counts.index and j in counts.columns) else 0
        legend_ax.text(j + 0.5, i + 0.5, f"{value}", ha = "center", va = "center", fontsize = 8, fontname = FONT_BODY)
legend_ax.set_xlim(0, 3)
legend_ax.set_ylim(0, 3)
legend_ax.text(1.5, -0.45, "Driving Access \u2192", ha = "center", fontsize = 9, fontname = FONT_BODY, fontweight = "bold")
legend_ax.text(-0.45, 1.5, "Walking Access \u2192", va = "center", rotation = 90, fontsize = 9, fontname = FONT_BODY, fontweight = "bold")

# Inset: Johannesburg, framed on shared CITY_EXTENTS.
jhb_xmin, jhb_xmax, jhb_ymin, jhb_ymax = CITY_EXTENTS["Johannesburg"]
axins_jhb = inset_axes(
    ax, width = "28%", height = "28%",
    loc = "upper right",
    bbox_to_anchor = (0, 0, 0.78, 1),
    bbox_transform = ax.transAxes,
)
jhb_clip = allaccess.cx[jhb_xmin:jhb_xmax, jhb_ymin:jhb_ymax]
jhb_clip.plot(color = jhb_clip["biv_color"], ax = axins_jhb, linewidth = 0.05, edgecolor = "white")
axins_jhb.set_aspect("equal")
axins_jhb.set_xlim(jhb_xmin, jhb_xmax)
axins_jhb.set_ylim(jhb_ymin, jhb_ymax)
#axins_jhb.set_facecolor("#fcffeb")
axins_jhb.set_xticks([])
axins_jhb.set_yticks([])
axins_jhb.set_title("Johannesburg / Pretoria", fontsize = 9, fontname = FONT_TITLE, fontweight = "bold")
for spine in axins_jhb.spines.values():
    spine.set_edgecolor("#442520")
    spine.set_linewidth(0.5)

# Inset: Durban, framed on shared CITY_EXTENTS.
dur_xmin, dur_xmax, dur_ymin, dur_ymax = CITY_EXTENTS["Durban"]
axins_dur = inset_axes(
    ax, width = "28%", height = "28%",
    loc = "lower left",
    bbox_to_anchor = (0.02, 0.04, 1, 1),
    bbox_transform = ax.transAxes,
)
dur_clip = allaccess.cx[dur_xmin:dur_xmax, dur_ymin:dur_ymax]
dur_clip.plot(color = dur_clip["biv_color"], ax = axins_dur, linewidth = 0.05, edgecolor = "white")
axins_dur.set_aspect("equal")
axins_dur.set_xlim(dur_xmin, dur_xmax)
axins_dur.set_ylim(dur_ymin, dur_ymax)
#axins_dur.set_facecolor("#fcffeb")
axins_dur.set_xticks([])
axins_dur.set_yticks([])
axins_dur.set_title("Durban", fontsize = 9, fontname = FONT_TITLE, fontweight = "bold")
for spine in axins_dur.spines.values():
    spine.set_edgecolor("#442520")
    spine.set_linewidth(0.5)

fig.savefig(FIG_DIR / "bivariate_walk_drive.png", dpi = 300, bbox_inches = "tight")
fig.savefig(FIG_DIR / "bivariate_walk_drive_transparent.png", dpi = 300, bbox_inches = "tight", transparent = True)
plt.show()